# LoRA Training — Mini-Batch Diagnostic Notebook

Replicates the `train_lora` Airflow DAG end-to-end in an interactive session.
Run each section individually to pinpoint exactly which step fails.

## Sections
1. Import Required Libraries
2. Configure LoRA Training Parameters
3. Load and Prepare Mini-Batch Dataset
4. Initialize Base Model and Apply LoRA Adapters
5. Define Training Loop
6. Run Training on Mini Batch
7. Evaluate and Log Training Metrics

## 1. Import Required Libraries

In [ ]:
import logging
import os
import sys
from pathlib import Path

# ── Project root resolution ───────────────────────────────────────────────
# Matches the PYTHONPATH the DAG sets for the subprocess.
# Works whether PROJECT_ROOT is set (Docker / CI) or running from the repo.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path("../..").resolve())).resolve()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)  # expose for Hydra oc.env resolver

for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")

# ── Standard training deps ─────────────────────────────────────────────────
import torch
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from transformers import AutoTokenizer

# ── Project modules ────────────────────────────────────────────────────────
from experiments.training.train_adapter.config import load_app_config, register_configs
from experiments.training.train_adapter.data_module import ArxivDataModule
from experiments.training.train_adapter.lit_module import PeftCausalLMModule
from experiments.training.train_adapter.modeling import build_model_and_tokenizer

register_configs()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print("All imports OK.")

## 2. Configure LoRA Training Parameters

Loads the same Hydra config the DAG uses (`experiment=train_adapter`), then
applies mini-batch overrides so it runs fast on a local machine.
Adjust any value here before proceeding.

In [ ]:
# ── Mini-batch overrides (tune as needed) ─────────────────────────────────
ACCELERATOR = "auto"  # "auto" | "gpu" | "cpu"
MINI_BATCHES = 5  # training steps before stopping
MINI_VAL = 2  # validation batches
MAX_EPOCHS = 1
BATCH_SIZE = 1
NUM_WORKERS = 0  # 0 = no multiprocessing (safer inside notebooks)

# Hydra overrides list — mirrors what you'd pass in the Airflow UI
HYDRA_OVERRIDES = [
    f"experiment.trainer.accelerator={ACCELERATOR}",
    f"experiment.trainer.max_epochs={MAX_EPOCHS}",
    f"experiment.data.batch_size={BATCH_SIZE}",
    f"experiment.data.num_workers={NUM_WORKERS}",
    # project_root is resolved from the env var set in cell 1
]

# ── Load Hydra config ──────────────────────────────────────────────────────
CONF_DIR = str(PROJECT_ROOT / "experiments" / "training" / "conf")

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=CONF_DIR, version_base=None):
    raw_cfg = compose(config_name="config", overrides=HYDRA_OVERRIDES)

app_cfg = load_app_config(raw_cfg)

print("─── paths ───────────────────────────────────────────────")
print(f"  project_root : {app_cfg.paths.project_root}")
print("─── model ───────────────────────────────────────────────")
print(f"  local_path   : {app_cfg.experiment.model.local_path}")
print(f"  load_in_4bit : {app_cfg.experiment.model.load_in_4bit}")
print("─── lora ────────────────────────────────────────────────")
print(f"  r            : {app_cfg.experiment.lora.r}")
print(f"  lora_alpha   : {app_cfg.experiment.lora.lora_alpha}")
print(f"  target_mods  : {app_cfg.experiment.lora.target_modules}")
print("─── data ────────────────────────────────────────────────")
print(f"  local_path   : {app_cfg.experiment.data.local_path}")
print(f"  batch_size   : {app_cfg.experiment.data.batch_size}")
print(f"  max_seq_len  : {app_cfg.experiment.data.max_seq_length}")
print("─── trainer ─────────────────────────────────────────────")
print(f"  accelerator  : {app_cfg.experiment.trainer.accelerator}")
print(f"  max_epochs   : {app_cfg.experiment.trainer.max_epochs}")
print(f"  precision    : {app_cfg.experiment.trainer.precision}")

## 3. Load and Prepare Mini-Batch Dataset

Uses the same `ArxivDataModule` the DAG runs, but calls `setup()` eagerly
so you can inspect the dataset and a sample batch before touching the model.

In [ ]:
# ── Verify dataset path exists before loading ──────────────────────────────
dataset_path = Path(app_cfg.experiment.data.local_path)
print(f"Dataset path : {dataset_path}")
print(f"Exists       : {dataset_path.exists()}")

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path}\n"
        "Run 'dvc pull assets/datasets/arxiv-summarization.dvc' to fetch it."
    )

# ── A tokenizer is needed by the DataModule ────────────────────────────────
# Load tokenizer only (cheap) so we can inspect batches without loading the full model.
model_path = Path(app_cfg.experiment.model.local_path)
print(f"\nTokenizer path : {model_path}")
print(f"Exists         : {model_path.exists()}")

tokenizer = AutoTokenizer.from_pretrained(str(model_path), use_fast=True, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Instantiate and setup DataModule ──────────────────────────────────────
data_cfg = app_cfg.experiment.data
datamodule = ArxivDataModule(tokenizer=tokenizer, data_cfg=data_cfg, shuffle=False)
datamodule.setup()

print(f"\nTrain split size : {len(datamodule.ds_train)}")
print(f"Val   split size : {len(datamodule.ds_val)}")

# ── Inspect first batch ────────────────────────────────────────────────────
train_loader = datamodule.train_dataloader()
first_batch = next(iter(train_loader))
print("\nFirst batch keys  :", list(first_batch.keys()))
for k, v in first_batch.items():
    print(f"  {k:15s}: shape={tuple(v.shape)}  dtype={v.dtype}")

## 4. Initialize Base Model and Apply LoRA Adapters

Calls the same `build_model_and_tokenizer` the DAG uses.
Prints trainable parameter count on success.

In [ ]:
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLoading model + applying LoRA adapters …")
model, tokenizer = build_model_and_tokenizer(app_cfg)
print("Model loaded and LoRA applied successfully.")

## 5. Build Training Loop via Config

Uses `hydra.utils.instantiate` to build the checkpoint callback and trainer
from the same `_target_`-based config that the DAG and CLI use.  Mini-batch
limits are passed as extra kwargs — no need to edit the config for a
diagnostic run.

In [ ]:
# ── Re-create DataModule with model's tokenizer (may differ from section 3) ─
datamodule = ArxivDataModule(tokenizer=tokenizer, data_cfg=app_cfg.experiment.data, shuffle=True)

# ── Lightning module ───────────────────────────────────────────────────────
train_cfg = app_cfg.experiment.training
sched_cfg = app_cfg.experiment.scheduler

lightning_module = PeftCausalLMModule(
    model=model,
    lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay,
    scheduler_cfg=sched_cfg.__dict__ if sched_cfg else None,
)

# ── Checkpoints — instantiated from the same config the DAG uses ──────────
artifacts_dir = PROJECT_ROOT / "artifacts" / "training"
artifacts_dir.mkdir(parents=True, exist_ok=True)

checkpoint_cb = instantiate(
    raw_cfg.experiment.callbacks.checkpoint,
    dirpath=str(artifacts_dir / "checkpoints"),
)

# ── Trainer — instantiated from config + mini-batch overrides ─────────────
trainer = instantiate(
    raw_cfg.experiment.trainer,
    callbacks=[checkpoint_cb],
    log_every_n_steps=1,  # log every step (tiny run)
    val_check_interval=1.0,  # validate once per epoch
    limit_train_batches=MINI_BATCHES,
    limit_val_batches=MINI_VAL,
    default_root_dir=str(artifacts_dir),
    enable_progress_bar=True,
)

print(f"Trainer configured: {MINI_BATCHES} train steps, {MINI_VAL} val steps.")
print(
    f"Accelerator : {app_cfg.experiment.trainer.accelerator}  |  Precision : {app_cfg.experiment.trainer.precision}"
)

## 6. Run Training on Mini Batch

Calls `trainer.fit()` exactly as the DAG's `run_training()` does.
Each step's loss is printed live via Lightning's progress bar.
Any exception here is the same exception that fails the Airflow task.

In [ ]:
import traceback

run_id = None
try:
    trainer.fit(lightning_module, datamodule=datamodule)
    # Retrieve MLflow run_id from the logger if attached
    if hasattr(trainer, "logger") and trainer.logger is not None:
        run_id = getattr(trainer.logger, "run_id", None)
    print(f"\nTraining finished successfully.  run_id={run_id}")
except Exception:
    print("\n" + "=" * 60)
    print("TRAINING FAILED — full traceback:")
    print("=" * 60)
    traceback.print_exc()
    print("=" * 60)

## 7. Evaluate and Log Training Metrics

Reads the logged metrics back from the Lightning trainer and prints a
summary comparable to what the DAG would record in MLflow.

In [ ]:
if trainer.state.status.value != "finished":
    print("Training did not finish — skipping metrics summary.")
else:
    # ── Logged metrics from Lightning callback_metrics ─────────────────────
    metrics = {k: float(v) for k, v in trainer.callback_metrics.items()}
    print("─── Callback metrics ────────────────────────────────────")
    for k, v in sorted(metrics.items()):
        print(f"  {k:<30s} {v:.6f}")

    # ── GPU memory snapshot ────────────────────────────────────────────────
    if torch.cuda.is_available():
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        print(f"\nPeak GPU memory allocated : {peak_mb:.1f} MB")

    # ── Quick manual validation on a single batch ──────────────────────────
    print("\n─── Manual validation pass ───────────────────────────────")
    lightning_module.eval()
    datamodule.setup("validate")
    val_loader = datamodule.val_dataloader()
    val_batch = next(iter(val_loader))

    device = next(lightning_module.parameters()).device
    val_batch = {k: v.to(device) for k, v in val_batch.items()}

    with torch.no_grad():
        outputs = lightning_module(
            val_batch["input_ids"],
            val_batch["attention_mask"],
            labels=val_batch["labels"],
        )
    print(f"  Manual val loss  : {outputs.loss.item():.6f}")
    print(f"  run_id           : {run_id or 'no MLflow logger attached'}")

    # ── Compare against expected DAG behaviour ────────────────────────────
    print("\n─── DAG equivalence check ────────────────────────────────")
    val_loss = metrics.get("val_loss", None)
    if val_loss is not None:
        status = "OK — val_loss present" if val_loss < 100 else "WARNING — val_loss unusually high"
        print(f"  val_loss = {val_loss:.4f}  →  {status}")
    else:
        print("  val_loss not found in metrics — check val_check_interval setting.")